In [0]:
from databricks.sdk import WorkspaceClient
import io

"""This notebook copies the data from /fixtures bundle folder to the landing volume in the Databricks workspace.
Then it creates a view referencing data uploded to the volume.
"""

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = "bronze"

root_path = dbutils.widgets.get("root_path")

file_relative_path = "/files/fixtures/taxi_zone_lookup.csv"
full_path = root_path + file_relative_path

In [0]:
w = WorkspaceClient()

file_content = w.workspace.download(full_path)
volume_path = f"/Volumes/{catalog}/landing/lookups/taxi_zone_lookup.csv"

w.files.upload(volume_path, io.BytesIO(file_content.read()), overwrite=True)

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.{schema}.taxi_zone_lookup_vw AS
    SELECT 
        LocationID as location_id,
        Borough as borough,
        Zone as zone,
        service_zone
    FROM read_files(
        '{volume_path}',
        format => 'csv',
        header => true,
        mode => 'FAILFAST')
""")


DataFrame[]